In [1]:
import os
import torch
import numpy as np
import pandas as pd
import yfinance as yf
import torch.nn as nn
from fredapi import Fred
from arch import arch_model
from dotenv import load_dotenv
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# --- Configuration ---
load_dotenv()
FRED_API_KEY = os.getenv('FRED_API_KEY')
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

TICKER = 'XLK'
START_DATE = '2000-01-01'
MACRO_TICKERS = {'VIX': '^VIX', 'Oil_WTI': 'CL=F'}
FRED_SERIES = {'CPI': 'CPIAUCSL', 'FedFunds': 'FEDFUNDS', 'Unemployment': 'UNRATE'}

def calculate_rolling_garch(returns, window=504, refit_every=21):
    """
    Calculates T+1 GARCH volatility using strictly historical data to prevent lookahead bias.
    Optimizes parameters every 21 days to save compute time, using fixed params in between.
    """
    print(f"Calculating Rolling GARCH (Window: {window} days)... This may take a minute.")
    garch_vol = pd.Series(index=returns.index, dtype=float)
    last_params = None

    # Start loop after our initial window
    for i in range(window, len(returns)):
        # Data up to day i-1 (strictly historical)
        train_data = returns.iloc[i-window : i]
        
        # Periodically optimize the GARCH parameters (e.g., once a month)
        if i == window or i % refit_every == 0:
            am = arch_model(train_data, vol='GARCH', p=1, q=1, mean='Constant')
            res = am.fit(disp='off', show_warning=False)
            last_params = res.params
            
        # Use the parameters to forecast T+1
        if last_params is not None:
            am = arch_model(train_data, vol='GARCH', p=1, q=1, mean='Constant')
            # .fix() applies previously optimized parameters instantly (zero-iteration)
            res = am.fix(last_params)
            forecasts = res.forecast(horizon=1, align='origin')
            # Extract the variance forecast for the next day
            garch_vol.iloc[i] = np.sqrt(forecasts.variance.iloc[-1, 0])
            
    return garch_vol

def build_feature_dataset(ticker, start_date, fred_api_key):
    """Fetches market data, calculates rolling volatility, and merges macro shocks."""
    print(f"Fetching Market Data for {ticker}...")
    df = yf.download(ticker, start=start_date, progress=False)
    
    # --- Data Safety Check ---
    if df.empty:
        raise ValueError(f"yfinance failed to download {ticker}. Try running: pip install --upgrade yfinance")
        
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # Base Returns
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
    df['Sq_Log_Return'] = df['Log_Return'] ** 2
    df = df.dropna().copy()

    # Volatility Features (Rolling)
    returns_scaled = df['Log_Return'] * 100
    # Using a 2-year (504 trading days) trailing window for the GARCH model
    df['Vol_GARCH'] = calculate_rolling_garch(returns_scaled, window=504, refit_every=21)
    
    # Drop the initial NaN period required to spin up the GARCH model
    df = df.dropna().copy()

    # Daily Macro Features
    print("Fetching Daily Macro (^VIX, Oil)...")
    daily_macro = yf.download(list(MACRO_TICKERS.values()), start=start_date, progress=False)['Close']
    if daily_macro.empty:
         raise ValueError("yfinance failed to download Macro variables.")
            
    daily_macro.columns = list(MACRO_TICKERS.keys())
    daily_macro['VIX_Change'] = daily_macro['VIX'].pct_change()
    daily_macro['Oil_Change'] = daily_macro['Oil_WTI'].pct_change()

    # Monthly FRED Features
    print("Fetching Monthly Macro (FRED)...")
    fred = Fred(api_key=fred_api_key)
    fred_data = {}
    for name, series_id in FRED_SERIES.items():
        series = fred.get_series(series_id, observation_start=start_date)
        macro_df = pd.DataFrame(series, columns=[name])
        if name == 'CPI':
            macro_df[f'{name}_MoM'] = macro_df[name].pct_change() * 100
        else:
            macro_df[f'{name}_Diff'] = macro_df[name].diff()
        fred_data[name] = macro_df

    monthly_macro = pd.concat(fred_data.values(), axis=1)

    # Merge Pipeline
    master_df = df.join(daily_macro, how='left')
    master_df = master_df.join(monthly_macro, how='left').ffill().dropna()
    
    print(f"Dataset completely built. Shape: {master_df.shape}")
    return master_df


In [2]:
def engineer_rolling_targets(df, window=252):
    """
    Classifies tomorrow's squared return based on the rolling distribution 
    of the past 'window' days to prevent lookahead bias.
    """
    print(f"Engineering rolling targets (Window: {window} days)...")
    df = df.copy()
    
    # Target is tomorrow's squared return
    df['Future_Sq_Return'] = df['Sq_Log_Return'].shift(-1)
    
    # Calculate rolling thresholds based on PAST data
    df['Rolling_Q50'] = df['Sq_Log_Return'].rolling(window=window).quantile(0.50)
    df['Rolling_Q85'] = df['Sq_Log_Return'].rolling(window=window).quantile(0.85)
    
    df = df.dropna()

    def classify_regime(row):
        val = row['Future_Sq_Return']
        if val <= row['Rolling_Q50']: return 0
        elif val <= row['Rolling_Q85']: return 1
        else: return 2

    df['Target_Regime'] = df.apply(classify_regime, axis=1).astype(int)
    
    # Clean up intermediate columns
    df = df.drop(columns=['Rolling_Q50', 'Rolling_Q85'])
    return df


# ==========================================
# Execution
# ==========================================
master_df = build_feature_dataset(TICKER, START_DATE, FRED_API_KEY)

# Assuming engineer_rolling_targets from the previous step is defined
master_df = engineer_rolling_targets(master_df, window=252)

Fetching Market Data for XLK...


Calculating Rolling GARCH (Window: 504 days)... This may take a minute.


Fetching Daily Macro (^VIX, Oil)...


Fetching Monthly Macro (FRED)...


Dataset completely built. Shape: (6120, 18)
Engineering rolling targets (Window: 252 days)...


In [3]:
class RegimeDataset(Dataset):
    def __init__(self, features, target, sequence_length):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.target = torch.tensor(target, dtype=torch.long)
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.features) - self.sequence_length

    def __getitem__(self, idx):
        return (self.features[idx : idx + self.sequence_length], 
                self.target[idx + self.sequence_length])


class HybridRegimeLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=2, num_classes=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc1 = nn.Linear(hidden_size, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :] # Decode last time step
        return self.fc2(self.relu(self.fc1(out)))


class RegimeModelTrainer:
    def __init__(self, model, lr=0.001, class_weights=None):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = model.to(self.device)
        
        if class_weights is not None:
            weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(self.device)
            self.criterion = nn.CrossEntropyLoss(weight=weights_tensor)
        else:
            self.criterion = nn.CrossEntropyLoss()
            
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)

    def train(self, train_loader, epochs=500, print_every=50):
        print("Starting Training...")
        self.model.train()
        for epoch in range(epochs):
            total_loss = 0
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                
                self.optimizer.zero_grad()
                loss = self.criterion(self.model(batch_x), batch_y)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()
                
            if (epoch + 1) % print_every == 0:
                print(f"Epoch [{epoch+1}/{epochs}] | Loss: {total_loss/len(train_loader):.4f}")

    def evaluate(self, test_loader):
        self.model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(self.device)
                _, preds = torch.max(self.model(batch_x), dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(batch_y.numpy())
                
        print("\n=== Evaluation Results ===")
        print(classification_report(all_targets, all_preds, 
                                    target_names=['Calm', 'Elevated', 'Shock'], 
                                    zero_division=0))

In [6]:
# Features to scale and feed into the network
feature_cols = ['Log_Return', 'Vol_GARCH', 'VIX_Change', 'Oil_Change', 'CPI_MoM', 'FedFunds_Diff']
target_col = 'Target_Regime'

# Chronological Split
split_idx = int(len(master_df) * 0.8)
train_df = master_df.iloc[:split_idx].copy()
test_df = master_df.iloc[split_idx:].copy()

# Scale Features
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols])
X_test = scaler.transform(test_df[feature_cols])
y_train = train_df[target_col].values
y_test = test_df[target_col].values

# Create DataLoaders
SEQ_LEN = 21
train_loader = DataLoader(RegimeDataset(X_train, y_train, SEQ_LEN), batch_size=32, shuffle=False)
test_loader = DataLoader(RegimeDataset(X_test, y_test, SEQ_LEN), batch_size=32, shuffle=False)

# Compute Imbalance Weights
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)

# Initialize and Train
lstm_model = HybridRegimeLSTM(input_size=len(feature_cols))
trainer = RegimeModelTrainer(lstm_model, class_weights=weights)

trainer.train(train_loader, epochs=500, print_every=50)
trainer.evaluate(test_loader)

Starting Training...


Epoch [50/750] | Loss: 0.8919


Epoch [100/750] | Loss: 0.7038


Epoch [150/750] | Loss: 0.5879


Epoch [200/750] | Loss: 0.4889


...

Epoch [550/750] | Loss: 0.2151


Epoch [600/750] | Loss: 0.1898


Epoch [650/750] | Loss: 0.1943


Epoch [700/750] | Loss: 0.1854


Epoch [750/750] | Loss: 0.1745

=== Evaluation Results ===
              precision    recall  f1-score   support

        Calm       0.50      0.47      0.48       568
    Elevated       0.33      0.35      0.34       396
       Shock       0.24      0.23      0.24       189

    accuracy                           0.39      1153
   macro avg       0.35      0.35      0.35      1153
weighted avg       0.40      0.39      0.39      1153



=== Evaluation Results === 500
              precision    recall  f1-score   support

        Calm       0.50      0.48      0.49       568
    Elevated       0.36      0.41      0.39       397
       Shock       0.25      0.23      0.24       188

    accuracy                           0.41      1153
   macro avg       0.37      0.37      0.37      1153
weighted avg       0.42      0.41      0.41      1153

=== Evaluation Results === 750
              precision    recall  f1-score   support

        Calm       0.50      0.47      0.48       568
    Elevated       0.33      0.35      0.34       396
       Shock       0.24      0.23      0.24       189

    accuracy                           0.39      1153
   macro avg       0.35      0.35      0.35      1153
weighted avg       0.40      0.39      0.39      1153


=== Evaluation Results === 200
              precision    recall  f1-score   support

        Calm       0.52      0.52      0.52       568
    Elevated       0.33      0.39      0.36       396
       Shock       0.26      0.16      0.20       189

    accuracy                           0.42      1153
   macro avg       0.37      0.36      0.36      1153
weighted avg       0.41      0.42      0.41      1153
